In [1]:
import argparse
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


# Dataset Path

In [2]:
TRAIN_PATH = "./dataset/train.csv"
TEST_PATH = "./dataset/test.csv"
SAMPLE_SUBMISSION_PATH = "./dataset/sample_submission.csv"

PLOT_PATH = "./plots"

# Load data

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)

os.makedirs(PLOT_PATH, exist_ok=True)

TARGET = "Will_Buy_EV"
ID_COL = "id"

print("=" * 70)
print("SHAPES")
print("=" * 70)
print(f"train: {train.shape}")
print(f"test : {test.shape}")
print(f"sample_submission: {sample_sub.shape}")

train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0}).astype(int)

SHAPES
train: (668665, 15)
test : (286571, 14)
sample_submission: (286571, 2)


# Basic structure

## Train Info

In [4]:
print("\n" + "=" * 70)
print("TRAIN HEAD")
print("=" * 70)
print(train.head())

print("\n" + "=" * 70)
print("TRAIN INFO (dtypes, non-null counts)")
print("=" * 70)
train.info()


TRAIN HEAD
   id  Age  Annual_Income_USD  Daily_Commute_km  Number_of_Cars_Owned  Charging_Stations_Near_Home  Charging_Stations_Near_Work  Environmental_Concern_Level  Gender City_Type Current_Car_Type  \
0   0   66            92887.0              23.4                     2                            3                            7                          1.0    Male  Suburban            Sedan   
1   1   38            30000.0               5.0                     1                            2                            2                          4.0    Male     Rural              SUV   
2   2   26            94389.0              36.8                     1                            8                           15                          5.0  Female     Urban            Sedan   
3   3   66            73580.0              23.7                     2                            6                            9                          3.0    Male  Suburban        Hatchback   
4   4   54   

## Test Info

In [5]:
print("\n" + "=" * 70)
print("TEST HEAD")
print("=" * 70)
print(test.head())

print("\n" + "=" * 70)
print("TEST INFO")
print("=" * 70)
test.info()


TEST HEAD
       id  Age  Annual_Income_USD  Daily_Commute_km  Number_of_Cars_Owned  Charging_Stations_Near_Home  Charging_Stations_Near_Work  Environmental_Concern_Level  Gender City_Type Current_Car_Type  \
0  668665   61            67725.0              16.9                     2                            7                            4                          4.0    Male  Suburban            Sedan   
1  668666   42           152835.0              41.9                     2                            9                            9                          4.0    Male     Urban              SUV   
2  668667   68            86877.0              53.3                     1                           10                           11                          4.0  Female     Urban            Sedan   
3  668668   39            46794.0              34.1                     2                            4                            8                          4.0  Female  Suburban            Sed

# Missing values

In [6]:
print("\n" + "=" * 70)
print("MISSING VALUES - TRAIN")
print("=" * 70)
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
print(missing_train if len(missing_train) else "No missing values in train.")

print("\n" + "=" * 70)
print("MISSING VALUES - TEST")
print("=" * 70)
missing_test = test.isnull().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)
print(missing_test if len(missing_test) else "No missing values in test.")


MISSING VALUES - TRAIN
No missing values in train.

MISSING VALUES - TEST
No missing values in test.


# Duplicates

In [7]:
print("\n" + "=" * 70)
print("DUPLICATES")
print("=" * 70)
print(f"Duplicate rows in train: {train.duplicated().shape[0] - train.drop_duplicates().shape[0]}")
print(f"Duplicate ids in train : {train[ID_COL].duplicated().sum()}")
print(f"Duplicate ids in test  : {test[ID_COL].duplicated().sum()}")


DUPLICATES
Duplicate rows in train: 0
Duplicate ids in train : 0
Duplicate ids in test  : 0


# Target distribution (class balance)

In [8]:
print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)
print('distribution count: ')
print(train[TARGET].value_counts())
print('distribution percestage: ')
print(train[TARGET].value_counts(normalize=True))

plt.figure(figsize=(5, 4))
sns.countplot(x=TARGET, data=train)
plt.title("Target class balance: Will_Buy_EV")
plt.tight_layout()
plt.savefig(f"{PLOT_PATH}/target_balance.png")
plt.close()


TARGET DISTRIBUTION
distribution count: 
Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64
distribution percestage: 
Will_Buy_EV
0    0.825355
1    0.174645
Name: proportion, dtype: float64


# Identify column types

In [9]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()

for c in [ID_COL, TARGET]:
    if c in num_cols:
        num_cols.remove(c)
    if c in cat_cols:
        cat_cols.remove(c)

print("\n" + "=" * 70)
print("COLUMN TYPES")
print("=" * 70)
print(f"Numeric columns ({len(num_cols)}): {num_cols}")
print(f"Categorical/object columns ({len(cat_cols)}): {cat_cols}")


COLUMN TYPES
Numeric columns (7): ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical/object columns (6): ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


# Numeric feature summary

In [10]:
print("\n" + "=" * 70)
print("NUMERIC SUMMARY - TRAIN")
print("=" * 70)
print(train[num_cols].describe().T)


NUMERIC SUMMARY - TRAIN
                                count          mean           std      min      25%      50%       75%       max
Age                          668665.0     47.039171     12.875448     25.0     36.0     47.0      58.0      69.0
Annual_Income_USD            668665.0  84769.266989  28648.029042  30000.0  67376.0  84880.0  102753.0  188549.0
Daily_Commute_km             668665.0     32.158298     18.730474      5.0     17.2     33.6      47.4      98.7
Number_of_Cars_Owned         668665.0      1.712626      0.729275      1.0      1.0      2.0       2.0       4.0
Charging_Stations_Near_Home  668665.0      4.960408      3.926843      0.0      2.0      4.0       7.0      14.0
Charging_Stations_Near_Work  668665.0      7.176314      5.186627      0.0      3.0      6.0      10.0      19.0
Environmental_Concern_Level  668665.0      2.935477      1.429119      1.0      2.0      3.0       4.0       5.0


# Categorical feature summary (unique values / cardinality)

In [11]:
print("\n" + "=" * 70)
print("CATEGORICAL SUMMARY - TRAIN")
print("=" * 70)
for c in cat_cols:
    n_unique = train[c].nunique()
    print(f"\n{c}  (unique values: {n_unique})")
    print(train[c].value_counts(dropna=False))



CATEGORICAL SUMMARY - TRAIN

Gender  (unique values: 3)
Gender
Male      367954
Female    295427
Other       5284
Name: count, dtype: int64

City_Type  (unique values: 3)
City_Type
Urban       289305
Suburban    255377
Rural       123983
Name: count, dtype: int64

Current_Car_Type  (unique values: 4)
Current_Car_Type
Sedan        303459
SUV          246545
Hatchback     79438
Truck         39223
Name: count, dtype: int64

Home_Charging_Possible  (unique values: 2)
Home_Charging_Possible
Yes    462677
No     205988
Name: count, dtype: int64

Subsidy_Available  (unique values: 2)
Subsidy_Available
Yes    419909
No     248756
Name: count, dtype: int64

Range_Anxiety_Level  (unique values: 3)
Range_Anxiety_Level
Low       603972
Medium     62499
High        2194
Name: count, dtype: int64


# Check train/test consistency for categorical columns(values in test not seen in train can break some encoders)

In [13]:
print("\n" + "=" * 70)
print("TRAIN vs TEST CATEGORY CONSISTENCY")
print("=" * 70)
for c in cat_cols:
    train_vals = set(train[c].dropna().unique())
    test_vals = set(test[c].dropna().unique())
    only_in_test = test_vals - train_vals
    only_in_train = train_vals - test_vals
    print(f"{c}: only_in_test={only_in_test if only_in_test else 'none'}, "
        f"only_in_train={only_in_train if only_in_train else 'none'}")
    
# none means both sets are identical




TRAIN vs TEST CATEGORY CONSISTENCY
Gender: only_in_test=none, only_in_train=none
City_Type: only_in_test=none, only_in_train=none
Current_Car_Type: only_in_test=none, only_in_train=none
Home_Charging_Possible: only_in_test=none, only_in_train=none
Subsidy_Available: only_in_test=none, only_in_train=none
Range_Anxiety_Level: only_in_test=none, only_in_train=none


# Distribution plots for numeric features (by target)

In [17]:
UNITS = {
    "Age": "years",
    "Annual_Income": "USD",
    "Daily_Commute_Distance": "km",
    "Vehicle_Price": "USD",
    "Fuel_Cost_Per_Year": "USD",
    "Charging_Station_Density": "stations / 100 km²",
}

# Friendly class labels for the target (comparison criterion)
TARGET_LABELS = {0: "No (0)", 1: "Yes (1)"}

# One plot per numeric feature
for c in num_cols:
    unit = UNITS.get(c, "")
    unit_str = f" ({unit})" if unit else ""

    fig, ax = plt.subplots(figsize=(8, 5))

    sns.histplot(
        data=train,
        x=c,
        hue=TARGET,
        kde=True,
        element="step",
        stat="count",              
        common_norm=False,         
        palette={0: "#4C72B0", 1: "#DD8452"},
        ax=ax,
    )

    # ---- axis labels with units ----
    ax.set_xlabel(f"{c}{unit_str}", fontsize=11)
    ax.set_ylabel("Count of customers", fontsize=11)

    # ---- title with per-class summary stats ----
    g = train.groupby(TARGET)[c]
    title_stats = "  |  ".join(
        f"{TARGET_LABELS[k]}: mean={g.mean()[k]:.2f}, median={g.median()[k]:.2f}"
        for k in sorted(g.mean().index)
    )
    ax.set_title(
        f"Distribution of {c}{unit_str} by {TARGET}\n{title_stats}",
        fontsize=12,
    )

    # ---- legend: name the comparison criterion explicitly ----
    leg = ax.legend(
        title=f"Comparison criterion:\n{TARGET}",
        labels=[TARGET_LABELS[k] for k in sorted(TARGET_LABELS)],
        loc="best",
        frameon=True,
    )
    leg.get_title().set_fontsize(10)

    ax.grid(alpha=0.3, linestyle="--")
    plt.tight_layout()

    # ---- save with a descriptive file name ----
    safe_name = c.replace(" ", "_").replace("/", "_")
    out_file = os.path.join(PLOT_PATH, f"hist_{safe_name}_by_{TARGET}.png")
    plt.savefig(out_file, dpi=130, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out_file}")

Saved: ./plots\hist_Age_by_Will_Buy_EV.png
Saved: ./plots\hist_Annual_Income_USD_by_Will_Buy_EV.png
Saved: ./plots\hist_Daily_Commute_km_by_Will_Buy_EV.png
Saved: ./plots\hist_Number_of_Cars_Owned_by_Will_Buy_EV.png
Saved: ./plots\hist_Charging_Stations_Near_Home_by_Will_Buy_EV.png
Saved: ./plots\hist_Charging_Stations_Near_Work_by_Will_Buy_EV.png
Saved: ./plots\hist_Environmental_Concern_Level_by_Will_Buy_EV.png


# Categorical vs target (buy rate per category)

In [18]:
print("\n" + "=" * 70)
print("BUY RATE BY CATEGORY")
print("=" * 70)
for c in cat_cols:
    rate = train.groupby(c)[TARGET].mean().sort_values(ascending=False)
    print(f"\n{c}:")
    print(rate)



BUY RATE BY CATEGORY

Gender:
Gender
Female    0.177641
Other     0.173732
Male      0.172253
Name: Will_Buy_EV, dtype: float64

City_Type:
City_Type
Rural       0.193389
Suburban    0.180936
Urban       0.161058
Name: Will_Buy_EV, dtype: float64

Current_Car_Type:
Current_Car_Type
SUV          0.180953
Hatchback    0.174299
Sedan        0.171967
Truck        0.156413
Name: Will_Buy_EV, dtype: float64

Home_Charging_Possible:
Home_Charging_Possible
Yes    0.195819
No     0.127085
Name: Will_Buy_EV, dtype: float64

Subsidy_Available:
Subsidy_Available
Yes    0.274695
No     0.005757
Name: Will_Buy_EV, dtype: float64

Range_Anxiety_Level:
Range_Anxiety_Level
Low       0.189027
Medium    0.041745
High      0.001367
Name: Will_Buy_EV, dtype: float64


# Correlation matrix (numeric features + target)

In [19]:
print("\n" + "=" * 70)
print("CORRELATION WITH TARGET (numeric features)")
print("=" * 70)
corr_with_target = train[num_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print(corr_with_target)

plt.figure(figsize=(8, 6))
sns.heatmap(train[num_cols + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix")
plt.tight_layout()
plt.savefig(f"{PLOT_PATH}/correlation_matrix.png")
plt.close()



CORRELATION WITH TARGET (numeric features)
Environmental_Concern_Level    0.464079
Annual_Income_USD              0.225729
Daily_Commute_km              -0.045866
Charging_Stations_Near_Home   -0.016353
Charging_Stations_Near_Work   -0.012229
Age                           -0.007450
Number_of_Cars_Owned           0.003896
Name: Will_Buy_EV, dtype: float64


# Outlier check (simple IQR based)

In [21]:
print("\n" + "=" * 70)
print("OUTLIER CHECK (IQR method)")
print("=" * 70)
for c in num_cols:
    q1, q3 = train[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((train[c] < lower) | (train[c] > upper)).sum()
    print(f"{c}: {n_outliers} potential outliers "
        f"(bounds: [{lower:.2f}, {upper:.2f}])")


OUTLIER CHECK (IQR method)
Age: 0 potential outliers (bounds: [3.00, 91.00])
Annual_Income_USD: 3678 potential outliers (bounds: [14310.50, 155818.50])
Daily_Commute_km: 41 potential outliers (bounds: [-28.10, 92.70])
Number_of_Cars_Owned: 13489 potential outliers (bounds: [-0.50, 3.50])
Charging_Stations_Near_Home: 0 potential outliers (bounds: [-5.50, 14.50])
Charging_Stations_Near_Work: 0 potential outliers (bounds: [-7.50, 20.50])
Environmental_Concern_Level: 0 potential outliers (bounds: [-1.00, 7.00])


# FEATURE SUMMARY TABLE

In [22]:
rows = []
for c in num_cols:
    rows.append({
        "feature": c,
        "type": "numeric",
        "n_unique": train[c].nunique(),
        "min": train[c].min(),
        "max": train[c].max(),
        "mean": train[c].mean(),
        "std": train[c].std(),
        "skew": train[c].skew(),
        "kurtosis": train[c].kurtosis(),
        "missing": train[c].isna().sum(),
    })
for c in cat_cols:
    rows.append({
        "feature": c,
        "type": "categorical",
        "n_unique": train[c].nunique(),
        "min": np.nan, "max": np.nan, "mean": np.nan, "std": np.nan,
        "skew": np.nan, "kurtosis": np.nan,
        "missing": train[c].isna().sum(),
    })

summary = pd.DataFrame(rows).sort_values("feature").reset_index(drop=True)
print(summary.to_string(index=False))
summary.to_csv(f"{PLOT_PATH}/feature_summary.csv", index=False)

                    feature        type  n_unique     min      max         mean          std      skew  kurtosis  missing
                        Age     numeric        45    25.0     69.0    47.039171    12.875448 -0.003786 -1.174029        0
          Annual_Income_USD     numeric     13214 30000.0 188549.0 84769.266989 28648.029042 -0.013086 -0.149234        0
Charging_Stations_Near_Home     numeric        15     0.0     14.0     4.960408     3.926843  0.698730 -0.482457        0
Charging_Stations_Near_Work     numeric        20     0.0     19.0     7.176314     5.186627  0.700188 -0.489245        0
                  City_Type categorical         3     NaN      NaN          NaN          NaN       NaN       NaN        0
           Current_Car_Type categorical         4     NaN      NaN          NaN          NaN       NaN       NaN        0
           Daily_Commute_km     numeric       805     5.0     98.7    32.158298    18.730474 -0.084589 -1.060918        0
Environmental_Concern_Le

# TARGET RATE BY BINNED NUMERIC FEATURE

In [23]:
N_BINS = 10
bin_rate_tables = {}

for c in num_cols:
    tmp = train[[c, TARGET]].copy()
    try:
        tmp["bin"] = pd.qcut(tmp[c], q=N_BINS, duplicates="drop")
    except ValueError:
        tmp["bin"] = pd.cut(tmp[c], bins=N_BINS)
    g = tmp.groupby("bin", observed=True)[TARGET].agg(["mean", "count"])
    bin_rate_tables[c] = g

    # Plot
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(g))
    ax.bar(x, g["mean"], color="#4C72B0")
    ax.axhline(train[TARGET].mean(), color="red", ls="--",
               label=f"overall mean = {train[TARGET].mean():.3f}")
    ax.set_xticks(x)
    ax.set_xticklabels([str(i) for i in g.index], rotation=45, ha="right")
    ax.set_xlabel(f"{c} (deciles)")
    ax.set_ylabel(f"P({TARGET}=1)")
    ax.set_title(f"Buy rate across deciles of {c}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{PLOT_PATH}/buyrate_bins_{c}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

    print(f"\n{c}:")
    print(g)


Age:
                    mean  count
bin                            
(24.999, 29.0]  0.189489  72933
(29.0, 34.0]    0.160169  69408
(34.0, 38.0]    0.170949  61837
(38.0, 43.0]    0.177092  74419
(43.0, 47.0]    0.173859  62401
(47.0, 51.0]    0.176670  61748
(51.0, 56.0]    0.200811  77954
(56.0, 60.0]    0.167178  56066
(60.0, 65.0]    0.159562  72931
(65.0, 69.0]    0.163987  58968

Annual_Income_USD:
                          mean  count
bin                                  
(29999.999, 45980.0]  0.044486  66942
(45980.0, 62671.0]    0.084593  66861
(62671.0, 71832.0]    0.106622  66834
(71832.0, 78915.0]    0.131650  66844
(78915.0, 84880.0]    0.156083  66977
(84880.0, 91619.0]    0.179741  67013
(91619.0, 96748.0]    0.197670  66616
(96748.0, 107798.0]   0.219553  66854
(107798.0, 122349.0]  0.289040  66970
(122349.0, 188549.0]  0.337313  66754

Daily_Commute_km:
                   mean   count
bin                            
(4.999, 21.9]  0.196327  201796
(21.9, 28.3]   0.18

# CATEGORICAL BUY RATE WITH 95% CI

In [24]:
overall = train[TARGET].mean()
cat_rate_tables = {}

for c in cat_cols:
    g = train.groupby(c)[TARGET].agg(["mean", "count"])
    g["se"] = np.sqrt(g["mean"] * (1 - g["mean"]) / g["count"])
    g["ci_lo"] = g["mean"] - 1.96 * g["se"]
    g["ci_hi"] = g["mean"] + 1.96 * g["se"]
    g = g.sort_values("mean", ascending=False)
    cat_rate_tables[c] = g

    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(g))
    ax.errorbar(x, g["mean"], yerr=1.96 * g["se"],
                fmt="o", capsize=4, color="#4C72B0")
    ax.axhline(overall, color="red", ls="--",
               label=f"overall = {overall:.3f}")
    ax.set_xticks(x)
    ax.set_xticklabels(g.index, rotation=20, ha="right")
    ax.set_ylabel(f"P({TARGET}=1)  ± 95% CI")
    ax.set_title(f"Buy rate by {c}")
    ax.legend()
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(f"{PLOT_PATH}/buyrate_cat_{c}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

    print(f"\n{c}:")
    print(g.round(4))


Gender:
          mean   count      se   ci_lo   ci_hi
Gender                                        
Female  0.1776  295427  0.0007  0.1763  0.1790
Other   0.1737    5284  0.0052  0.1635  0.1839
Male    0.1723  367954  0.0006  0.1710  0.1735

City_Type:
             mean   count      se   ci_lo   ci_hi
City_Type                                        
Rural      0.1934  123983  0.0011  0.1912  0.1956
Suburban   0.1809  255377  0.0008  0.1794  0.1824
Urban      0.1611  289305  0.0007  0.1597  0.1624

Current_Car_Type:
                    mean   count      se   ci_lo   ci_hi
Current_Car_Type                                        
SUV               0.1810  246545  0.0008  0.1794  0.1825
Hatchback         0.1743   79438  0.0013  0.1717  0.1769
Sedan             0.1720  303459  0.0007  0.1706  0.1733
Truck             0.1564   39223  0.0018  0.1528  0.1600

Home_Charging_Possible:
                          mean   count      se   ci_lo   ci_hi
Home_Charging_Possible                       

# CHI-SQUARE TEST OF INDEPENDENCE

In [25]:
from scipy.stats import chi2_contingency

chi_rows = []
for c in cat_cols:
    ct = pd.crosstab(train[c], train[TARGET])
    chi2, p, dof, expected = chi2_contingency(ct)
    # Cramér's V as effect size (0 = no assoc, 1 = perfect)
    n = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    chi_rows.append({
        "feature": c,
        "chi2": chi2,
        "p_value": p,
        "dof": dof,
        "cramers_v": cramers_v,
    })

chi_df = pd.DataFrame(chi_rows).sort_values("cramers_v", ascending=False)
print(chi_df.to_string(index=False))
chi_df.to_csv(f"{PLOT_PATH}/chi_square_results.csv", index=False)

               feature         chi2       p_value  dof  cramers_v
     Subsidy_Available 78382.499544  0.000000e+00    1   0.342378
   Range_Anxiety_Level  8981.912549  0.000000e+00    2   0.115899
Home_Charging_Possible  4671.041365  0.000000e+00    1   0.083580
             City_Type   742.830196 4.971328e-162    2   0.033330
      Current_Car_Type   173.662580  2.060262e-37    3   0.016116
                Gender    33.040870  6.687538e-08    2   0.007029


# MUTUAL INFORMATION WITH TARGET

In [26]:
from sklearn.feature_selection import mutual_info_classif

X_all = train[num_cols + cat_cols].copy()
# MI needs encoded input for categoricals
for c in cat_cols:
    X_all[c] = X_all[c].astype("category").cat.codes

mi = mutual_info_classif(X_all, train[TARGET], discrete_features="auto", random_state=42)
mi_series = pd.Series(mi, index=X_all.columns).sort_values(ascending=False)
print(mi_series)

fig, ax = plt.subplots(figsize=(8, 5))
mi_series.plot.barh(ax=ax, color="#4C72B0")
ax.invert_yaxis()
ax.set_xlabel("Mutual information with target")
ax.set_title("Feature importance (MI)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(f"{PLOT_PATH}/mutual_information.png", dpi=120, bbox_inches="tight")
plt.close()

Subsidy_Available              0.161615
Environmental_Concern_Level    0.147963
Home_Charging_Possible         0.099868
Range_Anxiety_Level            0.079894
Annual_Income_USD              0.053850
City_Type                      0.051106
Current_Car_Type               0.047025
Gender                         0.044208
Number_of_Cars_Owned           0.042123
Charging_Stations_Near_Home    0.006975
Charging_Stations_Near_Work    0.005488
Daily_Commute_km               0.005164
Age                            0.003796
dtype: float64


# FEATURE-FEATURE CORRELATION (numeric only) + VIF

In [27]:
corr = train[num_cols].corr()

# Only show pairs above a threshold
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
              .stack()
              .sort_values(key=np.abs, ascending=False))
print("Top correlated numeric pairs:")
print(pairs.head(15))

# VIF (multicollinearity)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_vif = add_constant(train[num_cols].sample(min(50000, len(train)), random_state=42))
vif = pd.Series(
    [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
    index=X_vif.columns
).drop("const")
print("\nVIF:")
print(vif.sort_values(ascending=False))

Top correlated numeric pairs:
Charging_Stations_Near_Home  Charging_Stations_Near_Work    0.511134
Annual_Income_USD            Environmental_Concern_Level    0.076405
Daily_Commute_km             Charging_Stations_Near_Home    0.045158
                             Charging_Stations_Near_Work    0.034986
                             Environmental_Concern_Level   -0.018868
Charging_Stations_Near_Home  Environmental_Concern_Level   -0.011332
Annual_Income_USD            Charging_Stations_Near_Home   -0.008646
Number_of_Cars_Owned         Charging_Stations_Near_Work    0.008611
                             Charging_Stations_Near_Home    0.008495
Annual_Income_USD            Daily_Commute_km               0.007994
Charging_Stations_Near_Work  Environmental_Concern_Level   -0.007833
Age                          Daily_Commute_km              -0.006538
                             Charging_Stations_Near_Work   -0.005855
                             Annual_Income_USD             -0.004559
    

# NUMERIC x CATEGORICAL INTERACTIONS

In [28]:
# Pick the strongest numeric feature (from MI / Pearson) and check
# whether its relationship with the target changes across categories.
strong_numeric = mi_series.head(2).index.tolist()
strong_numeric = [c for c in strong_numeric if c in num_cols]

for num_c in strong_numeric:
    for cat_c in cat_cols:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for cat_val, sub in train.groupby(cat_c):
            # bin numeric, compute rate within this category
            try:
                binned = pd.qcut(sub[num_c], q=6, duplicates="drop")
            except ValueError:
                continue
            rate = sub.groupby(binned, observed=True)[TARGET].mean()
            mids = [iv.mid for iv in rate.index]
            ax.plot(mids, rate.values, marker="o", label=str(cat_val))
        ax.axhline(train[TARGET].mean(), color="grey", ls="--", alpha=0.6)
        ax.set_xlabel(num_c)
        ax.set_ylabel(f"P({TARGET}=1)")
        ax.set_title(f"{num_c} vs target, split by {cat_c}")
        ax.legend(title=cat_c, fontsize=8)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        fname = f"{PLOT_PATH}/interaction_{num_c}_x_{cat_c}.png"
        plt.savefig(fname, dpi=110, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved {fname}")

Saved ./plots/interaction_Environmental_Concern_Level_x_Gender.png
Saved ./plots/interaction_Environmental_Concern_Level_x_City_Type.png
Saved ./plots/interaction_Environmental_Concern_Level_x_Current_Car_Type.png
Saved ./plots/interaction_Environmental_Concern_Level_x_Home_Charging_Possible.png
Saved ./plots/interaction_Environmental_Concern_Level_x_Subsidy_Available.png
Saved ./plots/interaction_Environmental_Concern_Level_x_Range_Anxiety_Level.png


# PAIRPLOT OF TOP FEATURES

In [29]:
PALETTE = {0: "#4C72B0", 1: "#DD8452"}

top_feats = mi_series.head(5).index.tolist()
sample = train.sample(min(20000, len(train)), random_state=42)

g = sns.pairplot(
    sample[top_feats + [TARGET]],
    hue=TARGET,
    palette=PALETTE,
    corner=True,
    plot_kws={"s": 8, "alpha": 0.4},
    diag_kind="kde",
)
g.fig.suptitle("Pairplot of top-5 features by MI", y=1.02)
g.savefig(f"{PLOT_PATH}/pairplot_top_features.png", dpi=110, bbox_inches="tight")
plt.close()

# BUY RATE PER SEGMENT (2-way)

In [30]:
from itertools import combinations

for c1, c2 in combinations(cat_cols, 2):
    ct = train.pivot_table(
        index=c1, columns=c2, values=TARGET, aggfunc="mean"
    )
    print(f"\n{c1} x {c2}: buy rate")
    print(ct.round(3))

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(ct, annot=True, fmt=".3f", cmap="viridis", ax=ax)
    ax.set_title(f"Buy rate: {c1} x {c2}")
    plt.tight_layout()
    plt.savefig(f"{PLOT_PATH}/heatmap_{c1}_x_{c2}.png", dpi=110, bbox_inches="tight")
    plt.close(fig)


Gender x City_Type: buy rate
City_Type  Rural  Suburban  Urban
Gender                           
Female     0.196     0.184  0.164
Male       0.191     0.179  0.159
Other      0.205     0.180  0.156

Gender x Current_Car_Type: buy rate
Current_Car_Type  Hatchback    SUV  Sedan  Truck
Gender                                          
Female                0.175  0.183  0.176  0.163
Male                  0.174  0.179  0.169  0.151
Other                 0.172  0.178  0.172  0.169

Gender x Home_Charging_Possible: buy rate
Home_Charging_Possible     No    Yes
Gender                              
Female                  0.130  0.198
Male                    0.125  0.194
Other                   0.134  0.193

Gender x Subsidy_Available: buy rate
Subsidy_Available     No    Yes
Gender                         
Female             0.006  0.278
Male               0.006  0.272
Other              0.007  0.271

Gender x Range_Anxiety_Level: buy rate
Range_Anxiety_Level   High    Low  Medium
Gender    

# MANN-WHITNEY U TEST

In [31]:
from scipy.stats import mannwhitneyu

mw_rows = []
for c in num_cols:
    a = train.loc[train[TARGET] == 1, c]
    b = train.loc[train[TARGET] == 0, c]
    stat, p = mannwhitneyu(a, b, alternative="two-sided")
    # Rank-biserial correlation as effect size
    n1, n2 = len(a), len(b)
    rbc = 1 - (2 * stat) / (n1 * n2)
    mw_rows.append({
        "feature": c,
        "U": stat,
        "p_value": p,
        "rank_biserial": rbc,
        "median_yes": a.median(),
        "median_no": b.median(),
    })

mw_df = pd.DataFrame(mw_rows).sort_values("rank_biserial", key=np.abs, ascending=False)
print(mw_df.to_string(index=False))
mw_df.to_csv(f"{PLOT_PATH}/mannwhitney_results.csv", index=False)

                    feature            U       p_value  rank_biserial  median_yes  median_no
Environmental_Concern_Level 5.436385e+10  0.000000e+00      -0.687043         5.0        2.0
          Annual_Income_USD 4.320426e+10  0.000000e+00      -0.340734     96167.0    82844.0
           Daily_Commute_km 3.004876e+10 1.722797e-291       0.067514        31.7       34.0
Charging_Stations_Near_Home 3.120050e+10  5.193597e-66       0.031773         4.0        4.0
Charging_Stations_Near_Work 3.153992e+10  2.346429e-30       0.021240         6.0        6.0
                        Age 3.186759e+10  2.606006e-09       0.011071        47.0       47.0
       Number_of_Cars_Owned 3.247792e+10  3.418005e-06      -0.007869         2.0        2.0


# TARGET ENCODING PREVIEW (with smoothing)

In [32]:
prior = train[TARGET].mean()
k = 20  # smoothing factor

for c in cat_cols:
    g = train.groupby(c)[TARGET].agg(["mean", "count"])
    g["smoothed"] = (g["count"] * g["mean"] + k * prior) / (g["count"] + k)
    g = g.sort_values("smoothed", ascending=False)
    print(f"\n{c} target encoding (smoothed, k={k}):")
    print(g.round(4))


Gender target encoding (smoothed, k=20):
          mean   count  smoothed
Gender                          
Female  0.1776  295427    0.1776
Other   0.1737    5284    0.1737
Male    0.1723  367954    0.1723

City_Type target encoding (smoothed, k=20):
             mean   count  smoothed
City_Type                          
Rural      0.1934  123983    0.1934
Suburban   0.1809  255377    0.1809
Urban      0.1611  289305    0.1611

Current_Car_Type target encoding (smoothed, k=20):
                    mean   count  smoothed
Current_Car_Type                          
SUV               0.1810  246545    0.1810
Hatchback         0.1743   79438    0.1743
Sedan             0.1720  303459    0.1720
Truck             0.1564   39223    0.1564

Home_Charging_Possible target encoding (smoothed, k=20):
                          mean   count  smoothed
Home_Charging_Possible                          
Yes                     0.1958  462677    0.1958
No                      0.1271  205988    0.1271

Sub

# USELESS FEATURE DETECTION

In [33]:
report = []
for c in num_cols + cat_cols:
    nunique = train[c].nunique(dropna=False)
    top_share = train[c].value_counts(normalize=True, dropna=False).iloc[0]
    report.append({
        "feature": c,
        "nunique": nunique,
        "top_value_share": round(top_share, 4),
        "is_constant": nunique == 1,
        "is_near_constant": top_share > 0.99,
    })
print(pd.DataFrame(report).to_string(index=False))

                    feature  nunique  top_value_share  is_constant  is_near_constant
                        Age       45           0.0266        False             False
          Annual_Income_USD    13214           0.0921        False             False
           Daily_Commute_km      805           0.2158        False             False
       Number_of_Cars_Owned        4           0.4461        False             False
Charging_Stations_Near_Home       15           0.1519        False             False
Charging_Stations_Near_Work       20           0.1193        False             False
Environmental_Concern_Level        5           0.2206        False             False
                     Gender        3           0.5503        False             False
                  City_Type        3           0.4327        False             False
           Current_Car_Type        4           0.4538        False             False
     Home_Charging_Possible        2           0.6919        Fals

# TRAIN/TEST DISTRIBUTION SHIFT (KS TEST)

In [35]:
from scipy.stats import ks_2samp

shift_rows = []
for c in num_cols:
    stat, p = ks_2samp(train[c], test[c])
    shift_rows.append({"feature": c, "KS_stat": stat, "p_value": p})

shift_df = pd.DataFrame(shift_rows).sort_values("KS_stat", ascending=False)
print(shift_df.to_string(index=False))

for c in num_cols:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.kdeplot(train[c], label="train", ax=ax, fill=True, alpha=0.3)
    sns.kdeplot(test[c],  label="test",  ax=ax, fill=True, alpha=0.3)
    ax.set_title(f"Train vs Test distribution: {c}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{PLOT_PATH}/shift_{c}.png", dpi=110, bbox_inches="tight")
    plt.close(fig)

                    feature  KS_stat  p_value
Charging_Stations_Near_Work 0.003005 0.053355
                        Age 0.002293 0.241771
       Number_of_Cars_Owned 0.001660 0.637171
          Annual_Income_USD 0.001639 0.653217
Charging_Stations_Near_Home 0.001579 0.698836
Environmental_Concern_Level 0.001271 0.901569
           Daily_Commute_km 0.001067 0.976165


# KDE OVERLAY: TRAIN (buy/no-buy) vs TEST

In [36]:
for c in num_cols:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.kdeplot(train.loc[train[TARGET] == 1, c], label="Train Yes",
                ax=ax, fill=True, alpha=0.35, color="#DD8452")
    sns.kdeplot(train.loc[train[TARGET] == 0, c], label="Train No",
                ax=ax, fill=True, alpha=0.35, color="#4C72B0")
    sns.kdeplot(test[c], label="Test", ax=ax, color="black", ls="--")
    ax.set_title(f"KDE: {c}")
    ax.set_xlabel(c)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{PLOT_PATH}/kde_{c}.png", dpi=110, bbox_inches="tight")
    plt.close(fig)

# UNIFIED SIGNAL TABLE

In [37]:
pearson = train[num_cols + [TARGET]].corr()[TARGET].drop(TARGET)

rows = []
for c in num_cols + cat_cols:
    if c in num_cols:
        pear = pearson.get(c, np.nan)
        mw = mw_df.set_index("feature").loc[c, "rank_biserial"] if c in mw_df["feature"].values else np.nan
        cram = np.nan
    else:
        pear = np.nan
        mw = np.nan
        cram = chi_df.set_index("feature").loc[c, "cramers_v"] if c in chi_df["feature"].values else np.nan
    rows.append({
        "feature": c,
        "type": "num" if c in num_cols else "cat",
        "mutual_info": mi_series.get(c, np.nan),
        "pearson": pear,
        "rank_biserial": mw,
        "cramers_v": cram,
    })

signal = pd.DataFrame(rows).sort_values("mutual_info", ascending=False)
print(signal.round(4).to_string(index=False))
signal.to_csv(f"{PLOT_PATH}/signal_summary.csv", index=False)

# Bar chart of MI for a visual
fig, ax = plt.subplots(figsize=(8, 6))
signal.set_index("feature")["mutual_info"].plot.barh(ax=ax, color="#4C72B0")
ax.invert_yaxis()
ax.set_title("Unified feature signal (Mutual Information)")
ax.set_xlabel("MI with target")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(f"{PLOT_PATH}/signal_summary.png", dpi=120, bbox_inches="tight")
plt.close()

                    feature type  mutual_info  pearson  rank_biserial  cramers_v
          Subsidy_Available  cat       0.1616      NaN            NaN     0.3424
Environmental_Concern_Level  num       0.1480   0.4641        -0.6870        NaN
     Home_Charging_Possible  cat       0.0999      NaN            NaN     0.0836
        Range_Anxiety_Level  cat       0.0799      NaN            NaN     0.1159
          Annual_Income_USD  num       0.0538   0.2257        -0.3407        NaN
                  City_Type  cat       0.0511      NaN            NaN     0.0333
           Current_Car_Type  cat       0.0470      NaN            NaN     0.0161
                     Gender  cat       0.0442      NaN            NaN     0.0070
       Number_of_Cars_Owned  num       0.0421   0.0039        -0.0079        NaN
Charging_Stations_Near_Home  num       0.0070  -0.0164         0.0318        NaN
Charging_Stations_Near_Work  num       0.0055  -0.0122         0.0212        NaN
           Daily_Commute_km 